# slide2pptx: train the element detector (YOLO)

**Runtime → Change runtime type → T4 GPU.** Needs the synthetic dataset made by
`colab_setup.ipynb` in Google Drive. Runs are saved to Drive, so a disconnect only
loses the current epoch: re-run cells 1–3, then the **resume** cell.

In [ ]:
# 1. Settings
REPO_URL  = "https://github.com/C00p1r/slide2pptx.git"
DRIVE_DIR = "/content/drive/MyDrive/slide2pptx"
DATA      = f"{DRIVE_DIR}/data/synth"          # from colab_setup.ipynb
MODEL     = "yolo11s.pt"                         # yolo11n.pt is ~2x faster, yolo11m.pt more accurate
EPOCHS    = 40
IMGSZ     = 1280                                 # small text/icons need large images
BATCH     = 8                                    # T4 (15 GB) at 1280 with yolo11s; lower if out of memory
RUN_NAME  = "y11s_1280_sim2real"
RUNS      = f"{DRIVE_DIR}/runs/detect"

In [ ]:
# 2. Mount Drive, clone/update the repo, install
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/slide2pptx'):
    !git clone {REPO_URL} /content/slide2pptx
else:
    !git -C /content/slide2pptx pull
%cd /content/slide2pptx
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Copy the dataset to local disk (reading thousands of files from Drive is very slow)
LOCAL = "/content/synth"
if not os.path.exists(LOCAL):
    !cp -r {DATA} {LOCAL}
!for s in train val test; do echo $s $(ls {LOCAL}/images/$s | wc -l); done

Quick check (optional): 1 epoch on 10% of the data, to make sure everything runs before the long job.

In [ ]:
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs 1 --fraction 0.1 \
    --imgsz {IMGSZ} --batch {BATCH} --project /content/runs --name smoke

In [ ]:
# 4. Train (yolo11s, 5k slides, 1280: roughly 3-5 min per epoch on a T4, so 40 epochs may need one resume)
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --imgsz {IMGSZ} --batch {BATCH} --project {RUNS} --name {RUN_NAME}

In [ ]:
# 4b. Resume after a disconnect (re-run cells 1-3 first)
!python -m detector.train --data {LOCAL}/data.yaml --project {RUNS} --name {RUN_NAME} --resume

In [ ]:
# 5. Per-class mAP on the synthetic test split -> Drive/results/detector/synth_test.md
W = f"{RUNS}/{RUN_NAME}/weights/best.pt"
!python -m detector.evaluate {W} --data {LOCAL}/data.yaml --split test --imgsz {IMGSZ} \
    --out {DRIVE_DIR}/results/detector --tag {RUN_NAME}_synth

## Real NotebookLM slides
Put the slide images in `DRIVE_DIR/data/real/images` (see `tools/extract_slides.py`).
Until they are hand-labelled (`docs/labelling.md`), we can only look at predictions.

In [ ]:
REAL = f"{DRIVE_DIR}/data/real"
!python -m detector.predict {W} {REAL}/images --out {DRIVE_DIR}/results/pred_real --imgsz {IMGSZ}
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"{DRIVE_DIR}/results/pred_real/*.jpg"))[:6]:
    display(Image(p, width=900))

In [ ]:
# Pre-labels for CVAT (docs/labelling.md, step 2)
!python tools/real_labels.py prelabel {W} --images {REAL}/images --out {REAL}/cvat_prelabels.zip

In [ ]:
# After labelling: build the real test set from the CVAT export and get real mAP
EXPORT = f"{REAL}/cvat_export.zip"     # upload your CVAT 'YOLO 1.1' export here
!python tools/real_labels.py build {EXPORT} --images {REAL}/images --out /content/real_yolo
!python -m detector.evaluate {W} --data /content/real_yolo/data.yaml --imgsz {IMGSZ} \
    --out {DRIVE_DIR}/results/detector --tag {RUN_NAME}_real

## Ablation for the report: does sim-to-real augmentation help?
Train the same model without the extra JPEG/blur/downscale augmentation and compare
real-slide mAP. This is the project's key result (sim-to-real gap).

In [ ]:
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --imgsz {IMGSZ} --batch {BATCH} --project {RUNS} --name {RUN_NAME}_noaug --no-sim2real

## Experiment: how much random layout should the training data have?
Plan and reasoning: `docs/experiments.md`. Short version: generate **two pools** once
(templates only, random layouts only), mix them into training sets with a share **R** of
random slides, train the **same small model** on each, and test every model on template
slides, random slides and the real NotebookLM slides. Everything except R stays fixed.

Run cells 1–2 first. The cells below skip work that is already on Drive, so after a
disconnect just run them again from the top of this section.

In [ ]:
# E1. Experiment settings (keep them fixed for every R)
POOL_N       = 2500                      # slides per pool -> 2000 train / 250 val / 250 test
RATIOS       = [0.0, 0.2, 0.5, 0.8, 1.0]
SWEEP_MODEL  = "yolo11n.pt"              # small model: 5 runs must fit the Colab budget
SWEEP_EPOCHS = 30
SWEEP_IMGSZ  = 1280
SWEEP_BATCH  = 16                        # yolo11n at 1280 on a T4; lower if out of memory
POOLS = f"{DRIVE_DIR}/data/pools"        # pool_tpl and pool_rand are saved here
SWEEP_RUNS = f"{DRIVE_DIR}/runs/ratio"
RESULTS = f"{DRIVE_DIR}/results/detector"

In [ ]:
# E2. Generate the two pools into Drive (once, ~35 min; CPU is enough).
# Same as colab_setup.ipynb: LibreOffice + fonts are needed only for this cell.
import os
if not all(os.path.exists(f"{POOLS}/{p}/data.yaml") for p in ("pool_tpl", "pool_rand")):
    !apt-get -qq update && apt-get -qq install -y libreoffice-impress poppler-utils fonts-liberation fonts-dejavu fonts-crosextra-carlito fonts-crosextra-caladea fonts-noto-cjk fonts-noto-color-emoji > /dev/null
    !fc-cache -f > /dev/null
    # different seeds, so slide i of each pool does not share a theme
    !python -m slidegen.generate --out {POOLS}/pool_tpl  --n {POOL_N} --random-ratio 0 --seed 0 --no-clean --jpg --workers 2
    !python -m slidegen.generate --out {POOLS}/pool_rand --n {POOL_N} --random-ratio 1 --seed 1 --no-clean --jpg --workers 2
!for p in pool_tpl pool_rand; do echo $p $(ls {POOLS}/$p/images/train | wc -l) train; done

In [ ]:
# E3. Copy pools to local disk and build one training set per R (every session, ~1 min)
for p in ("pool_tpl", "pool_rand"):
    if not os.path.exists(f"/content/{p}"):
        !cp -r {POOLS}/{p} /content/{p}
ratios_arg = ",".join(str(r) for r in RATIOS)
!python -m detector.mix --tpl /content/pool_tpl --rand /content/pool_rand --ratios {ratios_arg} --out /content/mix

In [ ]:
# E4. Train one model per R. Finished runs are skipped, interrupted ones are resumed.
# patience 0 = no early stopping, so every R trains for exactly SWEEP_EPOCHS.
import csv
def epochs_done(run):
    f = f"{run}/results.csv"
    return sum(1 for _ in open(f)) - 1 if os.path.exists(f) else 0

for r in RATIOS:
    name = f"r{r:.2f}"; run = f"{SWEEP_RUNS}/{name}"
    done = epochs_done(run)
    if done >= SWEEP_EPOCHS:
        print(name, "done"); continue
    if os.path.exists(f"{run}/weights/last.pt"):
        print(name, f"resuming after {done} epochs")
        !python -m detector.train --project {SWEEP_RUNS} --name {name} --resume
    else:
        !python -m detector.train --data /content/mix/{name}/data.yaml --model {SWEEP_MODEL} \
            --epochs {SWEEP_EPOCHS} --imgsz {SWEEP_IMGSZ} --batch {SWEEP_BATCH} --patience 0 --seed 0 \
            --project {SWEEP_RUNS} --name {name}

In [ ]:
# E5. Evaluate every model (last.pt: same number of epochs for all R, no picking by synthetic val)
#     on template test, random test and, once labelled, the real slides.
REAL_EXPORT = f"{DRIVE_DIR}/data/real/cvat_export.zip"   # see docs/labelling.md
TESTS = {"tpl": "/content/pool_tpl/data.yaml", "rand": "/content/pool_rand/data.yaml"}
if os.path.exists(REAL_EXPORT):
    if not os.path.exists("/content/real_yolo/data.yaml"):
        !python tools/real_labels.py build {REAL_EXPORT} --images {DRIVE_DIR}/data/real/images --out /content/real_yolo
    TESTS["real"] = "/content/real_yolo/data.yaml"
else:
    print("no real labels yet: evaluating on synthetic test sets only")
for r in RATIOS:
    w = f"{SWEEP_RUNS}/r{r:.2f}/weights/last.pt"
    if epochs_done(f"{SWEEP_RUNS}/r{r:.2f}") < SWEEP_EPOCHS:
        print(f"r{r:.2f} not finished, skipped"); continue
    for t, data in TESTS.items():
        tag = f"ratio_r{r:.2f}_{t}"
        if not os.path.exists(f"{RESULTS}/{tag}.json"):
            !python -m detector.evaluate {w} --data {data} --split test --imgsz {SWEEP_IMGSZ} --out {RESULTS} --tag {tag}

In [ ]:
# E6. Results table + plot -> Drive/results/detector/ratio_sweep.md and .png
!python -m detector.sweep {RESULTS} --prefix ratio
from IPython.display import Image, display
display(Image(f"{RESULTS}/ratio_sweep.png", width=900))

### Phase 2: confirm at full scale
Pick the best R on the **real** slides (or, before real labels exist, the R that
loses least on template slides while gaining on random ones). Retrain it and R = 0
with the final settings (yolo11s, more data, 40 epochs) to check the difference holds
for the model we will demo. Extend both pools first (same seeds, new indices). Do this only after E5–E6 are
complete: afterwards, cell E3 would build its sets from the larger pools.

In [ ]:
BEST_R = 0.2                       # set from the sweep
EXTRA = 2500                       # pools become 5000 each -> 4000 train each
!python -m slidegen.generate --out {POOLS}/pool_tpl  --n {EXTRA} --start {POOL_N} --random-ratio 0 --seed 0 --no-clean --jpg --workers 2
!python -m slidegen.generate --out {POOLS}/pool_rand --n {EXTRA} --start {POOL_N} --random-ratio 1 --seed 1 --no-clean --jpg --workers 2
!rm -rf /content/pool_tpl /content/pool_rand && cp -r {POOLS}/pool_tpl /content/ && cp -r {POOLS}/pool_rand /content/
!python -m detector.mix --tpl /content/pool_tpl --rand /content/pool_rand --ratios 0,{BEST_R} --out /content/mix_full
for r in (0.0, BEST_R):
    name = f"r{r:.2f}"
    !python -m detector.train --data /content/mix_full/{name}/data.yaml --model yolo11s.pt --epochs 40 \
        --imgsz 1280 --batch 8 --project {SWEEP_RUNS} --name full_{name}